In [ ]:
%pip install -q kagglehub libreyolo
%pip install -q --upgrade jupyter ipywidgets
# !git clone https://github.com/LuisPeregrina/gdl-atsc-anti-spillback.git
# !mv gdl-atsc-anti-spillback/* .

In [ ]:
import shutil
from pathlib import Path
from time import sleep

import kagglehub
import yaml
from libreyolo import LibreYOLO
from libreyolo.training import TrainEndEvent, TrainEpochEvent, TrainStartEvent, TrainExceptionEvent

from tools.mtid_split_yolo import split_dataset

In [ ]:
MODEL_NAME = "LibreYOLO9t"

DATASET_PATH = Path.cwd() / "dataset"
DATASET_NAME = "andreasmoegelmose/multiview-traffic-intersection-dataset"
EPOCHS = 100

# Configs per model
models = yaml.safe_load(Path("models.yaml").read_text())["models"]
model = next((m for m in models if m["name"] == MODEL_NAME), None)
IMAGE_SIZE = int(model["image_size"])
BATCH_SIZE = int(model["batch_size"])


In [ ]:
dataset_path = kagglehub.dataset_download(DATASET_NAME, output_dir=str(DATASET_PATH))
# If "Using Colab cache for faster access to the 'multiview-traffic-intersection-dataset' dataset." we need to copy from cache
#!cp -r {dataset_path} {DATASET_PATH}

In [ ]:
yaml_path = split_dataset(dataset_path)

Some utils so that we can interrupt and get the last weights

In [ ]:
# Logger
class RunLog:
    def copy_last(self, event: TrainEpochEvent) -> None:
        fname = "last.pt"
        event_last_pt = Path(event.save_dir) / "weights" / fname
        if not event_last_pt.exists():
            print(f"Warning: {event_last_pt} does not exist, skipping copy.")
            return
        print(f"Copying {event_last_pt} to {fname}")
        shutil.copy(event_last_pt, fname)

 
    def on_train_epoch_end(self, event: TrainEpochEvent) -> None:
        if event.is_best:
            print(f"new best at epoch {event.epoch}: {event.best_metric}")

        self.copy_last(event)
        print("Sleeping 3 minutes to cool down...")
        sleep(60*3)

In [ ]:
model = LibreYOLO("LibreFOMOs-point.pt")  # last.pt
last_weights_path = None
model.train(
        data=yaml_path,
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        workers=0,
        callbacks=RunLog(),
        # resume=True
    )